# 01_load_check

東京電力PowerGrid「でんき予報」過去実績CSV (2021〜2025) の読み込み確認。


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT/"data"/"raw"

print("RAW_DIR:",RAW_DIR)

RAW_DIR: c:\Users\koudai.sugawara\power-demand-analysis\data\raw


In [2]:
df_2024 = pd.read_csv(
    RAW_DIR/"juyo-2024.csv",
    encoding = "shift_jis",
    skiprows = 2
)

print("shape:",df_2024.shape)
print("columns:",list(df_2024.columns))
df_2024.head()

shape: (8784, 3)
columns: ['DATE', 'TIME', '実績(万kW)']


,DATE,TIME,実績(万kW)
0,2024/1/1,0:00,2402
1,2024/1/1,1:00,2286
2,2024/1/1,2:00,2243
3,2024/1/1,3:00,2229
4,2024/1/1,4:00,2229


In [3]:
df_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   DATE     8784 non-null   str  
 1   TIME     8784 non-null   str  
 2   実績(万kW)  8784 non-null   int64
dtypes: int64(1), str(2)
memory usage: 206.0 KB


## 5年分をまとめて読み込み

各年のCSVを読み込み、datetime列を作って縦に連結する。


In [4]:
def load_year(year):
    path = RAW_DIR/f"juyo-{year}.csv"
    df = pd.read_csv(path, encoding="shift_jis", skiprows=2)
    df = df.rename(columns = {"実績(万kW)":"demand"})

    df["datetime"] = pd.to_datetime(
        df["DATE"] + " " + df["TIME"],
        format="%Y/%m/%d %H:%M"
    )

    df["year"] = year
    return df[["datetime","year","demand"]]

years = [2021,2022,2023,2024,2025]
df_all = pd.concat([load_year(y) for y in years], ignore_index=True)

print("total_shape:", df_all.shape)
df_all.head()



total_shape: (39912, 3)


,datetime,year,demand
0,2021-01-01 00:00:00,2021,3184
1,2021-01-01 01:00:00,2021,2978
2,2021-01-01 02:00:00,2021,2834
3,2021-01-01 03:00:00,2021,2743
4,2021-01-01 04:00:00,2021,2697


## 年ごとのサマリ

行数・欠損・期間を年単位で集計して、データの揃い具合を確認する。


In [8]:
summary = (
    df_all.groupby("year")
    .agg(
        rows=("demand","size"),
        n_missing=("demand",lambda s: s.isna().sum()), 
        start=("datetime","min"),
        end=("datetime","max")
        
    )
    .reset_index()
)
summary

,year,rows,n_missing,start,end
0,2021,8760,0,2021-01-01,2021-12-31 23:00:00
1,2022,8760,0,2022-01-01,2022-12-31 23:00:00
2,2023,8760,0,2023-01-01,2023-12-31 23:00:00
3,2024,8784,0,2024-01-01,2024-12-31 23:00:00
4,2025,4848,0,2025-01-01,2025-07-21 23:00:00
